# Modeling 6: 10,000 California red/white training reviews

This stage retains Model 5's first 1,000 training reviews, adds 9,000 more, and uses the same full California red/white validation set. The three TF-IDF models run locally. **DistilBERT trains on a Google Colab GPU; its compact results ZIP is then loaded here so all four models appear in one comparison.** The dummy baseline is omitted.

**Workflow**

1. Run Sections 1–3 locally to get the TF-IDF results.
2. Open this same notebook in Google Colab, select **Runtime → Change runtime type → T4 GPU**, and upload `train.parquet` and `validation.parquet` when prompted. Run Sections 1–4. Section 4 trains DistilBERT and downloads `modeling_6_distilbert_results.zip` containing predictions and metrics, not model weights.
3. Put that ZIP beside this notebook locally. Run Sections 1–3, then Section 5. It verifies that Colab used the exact same selected training and validation rows before adding DistilBERT to the comparison table. Section 4 automatically skips outside Colab.

Only `description` is a model feature. Metadata selects the California population and supplies red/white labels. `test.parquet` remains reserved for final evaluation.

## 1. Load the saved partitions

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

SEED = 42
LABELS = ['red', 'white']
if not all(Path(name).exists() for name in ['train.parquet', 'validation.parquet']):
    try:
        from google.colab import files
    except ImportError as exc:
        raise FileNotFoundError('Place train.parquet and validation.parquet beside this notebook.') from exc
    print('Upload train.parquet and validation.parquet together.')
    files.upload()
assert all(Path(name).exists() for name in ['train.parquet', 'validation.parquet'])
saved_train = pd.read_parquet('train.parquet')
saved_validation = pd.read_parquet('validation.parquet')
assert saved_train.index.intersection(saved_validation.index).empty
print(f'Saved partitions: {len(saved_train):,} train; {len(saved_validation):,} validation')

Saved partitions: 101,506 train; 33,835 validation


## 2. Retain Model 5's 1,000 reviews and add 9,000 more

Use the same California red/white selection and the same saved validation rows as Model 5. Normalize review text, exclude conflicting labels and duplicate descriptions, and remove training copies of any text appearing in validation. First reproduce Model 5's 500-per-color draw, then add 4,500 per color from unused training rows.

In [3]:
N_TRAIN = 10_000
MODEL_NUMBER = 6

def california_pool(frame):
    pool = frame.loc[frame['province'].eq('California') & frame['color'].isin(LABELS)].dropna(subset=['description', 'color']).copy()
    pool['description_key'] = pool['description'].str.replace(r'\s+', ' ', regex=True).str.strip().str.casefold()
    pool = pool.loc[pool['description_key'].ne('')]
    conflicts = pool.groupby('description_key')['color'].nunique()
    pool = pool.loc[~pool['description_key'].isin(conflicts[conflicts > 1].index)]
    return pool.drop_duplicates('description_key')

train_pool = california_pool(saved_train)
validation_all = california_pool(saved_validation)
validation_keys = set(validation_all['description_key'])
cross_split_keys = set(train_pool['description_key']) & validation_keys
train_pool = train_pool.loc[~train_pool['description_key'].isin(cross_split_keys)].copy()
assert set(train_pool['description_key']).isdisjoint(validation_keys)
assert train_pool.index.intersection(validation_all.index).empty
assert all(train_pool['color'].eq(color).sum() >= 5_000 for color in LABELS)

model5_parts = []
additional_parts = []
for color in LABELS:
    color_pool = train_pool.loc[train_pool['color'].eq(color)]
    original_500 = color_pool.sample(n=500, random_state=SEED)  # Model 5's draw
    remaining = color_pool.loc[~color_pool.index.isin(original_500.index)]
    model5_parts.append(original_500)
    additional_parts.append(remaining.sample(n=4_500, random_state=SEED))
model5_training_indices = set(pd.concat(model5_parts).index)
train_sample = pd.concat([*model5_parts, *additional_parts]).sample(frac=1, random_state=SEED)
assert len(train_sample) == N_TRAIN and train_sample.index.is_unique
assert model5_training_indices.issubset(set(train_sample.index))
assert set(train_sample['description_key']).isdisjoint(validation_keys)
print('Model 5 training rows retained:', len(model5_training_indices))
print('Cross-split normalized descriptions excluded from training:', len(cross_split_keys))
display(pd.DataFrame({'training': train_sample['color'].value_counts(), 'validation': validation_all['color'].value_counts()}).reindex(LABELS))
print('Validation uses every eligible California red/white review:', len(validation_all))

Model 5 training rows retained: 1000
Cross-split normalized descriptions excluded from training: 0


,training,validation
color,,
red,5000,6791
white,5000,2614


Validation uses every eligible California red/white review: 9405


## 3. Compare three TF-IDF classifiers

Keep Model 5's unigram/bigram TF-IDF settings and classifier settings. Fit the vectorizer separately on each model's training reviews; only the training size changes. Report training and validation accuracy, macro F1, per-class precision/recall, and confusion matrices.

In [4]:
def text_features():
    return TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=10_000, sublinear_tf=True)

models = {
    'Naive Bayes': make_pipeline(text_features(), MultinomialNB(alpha=1.0)),
    'Logistic regression': make_pipeline(text_features(), LogisticRegression(C=1.0, max_iter=2000, random_state=SEED)),
    'Linear SVM': make_pipeline(text_features(), LinearSVC(C=1.0, max_iter=5000, random_state=SEED)),
}
X_train, y_train = train_sample['description'], train_sample['color']
X_val, y_val = validation_all['description'], validation_all['color']
score_rows = []
validation_predictions = {}

def record_scores(name, train_true, train_pred, val_true, val_pred):
    for split, actual, predicted in [('training', train_true, train_pred), ('validation', val_true, val_pred)]:
        score_rows.append({'model': name, 'split': split, 'n': len(actual), 'accuracy': accuracy_score(actual, predicted), 'macro_f1': f1_score(actual, predicted, labels=LABELS, average='macro', zero_division=0)})
    validation_predictions[name] = np.asarray(val_pred)

for name, model in models.items():
    model.fit(X_train, y_train)
    record_scores(name, y_train, model.predict(X_train), y_val, model.predict(X_val))
    print('Finished', name)

Finished Naive Bayes
Finished Logistic regression
Finished Linear SVM


In [5]:
def show_comparison():
    scores = pd.DataFrame(score_rows).pivot(index='model', columns='split', values=['accuracy', 'macro_f1'])
    scores.columns = [f'{metric}_{split}' for metric, split in scores.columns]
    scores['accuracy_gap'] = scores['accuracy_training'] - scores['accuracy_validation']
    scores['macro_f1_gap'] = scores['macro_f1_training'] - scores['macro_f1_validation']
    display(scores.sort_values('macro_f1_validation', ascending=False).round(3))
    for name, predicted in validation_predictions.items():
        print(f'\n{name}: validation report')
        print(classification_report(y_val, predicted, labels=LABELS, zero_division=0, digits=3))
        display(pd.DataFrame(confusion_matrix(y_val, predicted, labels=LABELS), index=['actual red', 'actual white'], columns=['predicted red', 'predicted white']))
    return scores

comparison = show_comparison()

,accuracy_training,accuracy_validation,macro_f1_training,macro_f1_validation,accuracy_gap,macro_f1_gap
model,,,,,,
Linear SVM,1.000,0.989,1.000,0.987,0.010,0.013
Logistic regression,0.997,0.989,0.997,0.987,0.008,0.011
Naive Bayes,0.992,0.987,0.991,0.984,0.004,0.007



Naive Bayes: validation report
              precision    recall  f1-score   support

         red      0.996     0.986     0.991      6791
       white      0.966     0.989     0.977      2614

    accuracy                          0.987      9405
   macro avg      0.981     0.988     0.984      9405
weighted avg      0.987     0.987     0.987      9405



,predicted red,predicted white
actual red,6699,92
actual white,29,2585



Logistic regression: validation report
              precision    recall  f1-score   support

         red      0.996     0.989     0.992      6791
       white      0.971     0.990     0.981      2614

    accuracy                          0.989      9405
   macro avg      0.984     0.990     0.987      9405
weighted avg      0.989     0.989     0.989      9405



,predicted red,predicted white
actual red,6714,77
actual white,25,2589



Linear SVM: validation report
              precision    recall  f1-score   support

         red      0.996     0.989     0.993      6791
       white      0.973     0.989     0.981      2614

    accuracy                          0.989      9405
   macro avg      0.984     0.989     0.987      9405
weighted avg      0.989     0.989     0.989      9405



,predicted red,predicted white
actual red,6719,72
actual white,28,2586


### Observed local TF-IDF results

These results were reproduced on **10,000 balanced California training reviews** (5,000 red; 5,000 white) and the unchanged **9,405-review California validation set** (6,791 red; 2,614 white). All 1,000 Model 5 training reviews are retained. DistilBERT has not been run here.

| Model | Training accuracy | Validation accuracy | Validation macro F1 | Macro F1 gap |
|---|---:|---:|---:|---:|
| Naive Bayes | 99.15% | 98.71% | 0.984 | 0.007 |
| Logistic regression | 99.71% | 98.92% | 0.987 | 0.011 |
| Linear SVM | 99.98% | 98.94% | 0.987 | 0.013 |

All three have higher validation macro F1 and smaller gaps than in Model 5. The validation gain is modest because Model 5 was already strong. The SVM and logistic regression scores are very close; inspect class-level errors and the final holdout before choosing between them.

### Bias–variance decisions

- **Naive Bayes:** Compare its Model 5 and Model 6 validation macro F1 and training gap. If more data helps but a gap remains, test smoothing (`alpha`) or feature settings while holding this split fixed.
- **Logistic regression:** If the larger sample improves validation and reduces the gap, data was the useful intervention. If training remains high but validation plateaus, try a smaller `C` for stronger regularization.
- **Linear SVM:** If its gap persists, try a smaller `C`; judge the change by validation macro F1, not merely by narrowing the gap. Inspect errors by varietal.
- **DistilBERT:** Compare against the simpler models after training. More data may help it most, but check validation macro F1 by epoch before changing epochs or weight decay. A higher-capacity model is not automatically preferable.

Validation remains naturally imbalanced (more red than white), while training is balanced. Use macro F1 and class-level errors as primary guides. A training–validation gap alone is not a formal bias/variance estimate.

## 4. Train DistilBERT on Google Colab and download results

This section runs only inside Colab and requires a GPU. It uses the same 10,000 training descriptions and 9,405 validation descriptions as the TF-IDF models above. The pretrained DistilBERT weights are fine-tuned for three epochs, with the best checkpoint selected by validation macro F1. After training, the notebook exports row-level training and validation predictions, training history, and data fingerprints in a small ZIP. The model weights remain in Colab; the ZIP has everything needed to compare scores here.

The next cell installs the BERT dependencies only in Colab. After its first install or upgrade, **restart the Colab runtime and rerun from the top** before importing Transformers. Package upgrades in a live kernel can cause stale-import errors.

In [5]:
from importlib.util import find_spec
from importlib import metadata
IN_COLAB = find_spec('google.colab') is not None
if IN_COLAB:
    # Install only in Colab; stop if package versions changed so imports use a fresh runtime.
    packages = ('torch', 'transformers', 'accelerate', 'datasets', 'pyarrow')
    def installed_version(package):
        try:
            return metadata.version(package)
        except metadata.PackageNotFoundError:
            return None
    before = {package: installed_version(package) for package in packages}
    get_ipython().run_line_magic('pip', '-q install -U "transformers==4.57.6" "accelerate>=1,<2" "datasets>=4,<5"')
    after = {package: installed_version(package) for package in packages}
    for package, version in after.items():
        print(f'{package}: {version or "missing"}')
    if before != after:
        raise RuntimeError('Colab packages changed. Restart the runtime and rerun this notebook from the top.')
else:
    print('Local run: Colab training will be skipped. Use Section 5 to import its results ZIP.')

torch: 2.12.0
transformers: 4.57.6
accelerate: 1.15.0
datasets: 4.8.5
pyarrow: 23.0.1


In [ ]:
if IN_COLAB:
    import torch
    from torch.utils.data import Dataset
    from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding, Trainer, TrainingArguments, set_seed


    set_seed(SEED)
    print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else ('Apple MPS' if torch.backends.mps.is_available() else 'CPU'))
    MODEL_NAME = 'distilbert/distilbert-base-uncased'
    label2id = {'red': 0, 'white': 1}
    id2label = {v: k for k, v in label2id.items()}
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

    class ReviewDataset(Dataset):
        def __init__(self, frame):
            self.encodings = tokenizer(frame['description'].tolist(), truncation=True, max_length=128)
            self.labels = frame['color'].map(label2id).astype(int).tolist()
        def __len__(self):
            return len(self.labels)
        def __getitem__(self, index):
            return {**{key: torch.tensor(values[index]) for key, values in self.encodings.items()}, 'labels': torch.tensor(self.labels[index])}

    bert_train = ReviewDataset(train_sample)
    bert_val = ReviewDataset(validation_all)
else:
    print("Skipped DistilBERT training: run this section on a Colab GPU.")

In [ ]:
if IN_COLAB:
    def bert_metrics(result):
        predicted = result.predictions.argmax(axis=-1)
        return {'accuracy': accuracy_score(result.label_ids, predicted), 'macro_f1': f1_score(result.label_ids, predicted, labels=[0, 1], average='macro', zero_division=0)}

    bert_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2, id2label=id2label, label2id=label2id)
    use_bf16 = torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8 and torch.cuda.is_bf16_supported()
    bert_args = TrainingArguments(
        output_dir=f'modeling_{MODEL_NUMBER}_bert_checkpoints',
        learning_rate=2e-5, per_device_train_batch_size=16, per_device_eval_batch_size=32,
        num_train_epochs=3, weight_decay=0.01,
        eval_strategy='epoch', save_strategy='epoch', load_best_model_at_end=True,
        metric_for_best_model='macro_f1', greater_is_better=True, save_total_limit=1,
        fp16=torch.cuda.is_available() and not use_bf16, bf16=use_bf16,
        report_to='none', seed=SEED,
    )
    bert_trainer = Trainer(
        model=bert_model, args=bert_args, train_dataset=bert_train, eval_dataset=bert_val,
        processing_class=tokenizer, data_collator=DataCollatorWithPadding(tokenizer),
        compute_metrics=bert_metrics,
    )
    bert_trainer.train()
else:
    print("Skipped DistilBERT training: run this section on a Colab GPU.")

In [ ]:
if IN_COLAB:
    import hashlib
    import io
    import json
    import zipfile
    from google.colab import files

    def frame_fingerprint(frame):
        rows = [[int(row_id), str(row['color']), str(row['description'])]
                for row_id, row in frame.sort_index().iterrows()]
        payload = json.dumps(rows, ensure_ascii=False, separators=(',', ':')).encode('utf-8')
        return hashlib.sha256(payload).hexdigest()

    train_output = bert_trainer.predict(bert_train)
    val_output = bert_trainer.predict(bert_val)
    train_pred = [id2label[int(i)] for i in train_output.predictions.argmax(axis=-1)]
    val_pred = [id2label[int(i)] for i in val_output.predictions.argmax(axis=-1)]
    train_table = pd.DataFrame({'row_id': train_sample.index.astype(int),
                                'actual_color': train_sample['color'].to_numpy(),
                                'predicted_color': train_pred})
    val_table = pd.DataFrame({'row_id': validation_all.index.astype(int),
                              'actual_color': validation_all['color'].to_numpy(),
                              'predicted_color': val_pred})
    export_metadata = {
        'experiment': 'modeling_6_distilbert',
        'model': MODEL_NAME,
        'seed': SEED,
        'training_rows': len(train_sample),
        'validation_rows': len(validation_all),
        'training_fingerprint': frame_fingerprint(train_sample),
        'validation_fingerprint': frame_fingerprint(validation_all),
        'best_checkpoint': bert_trainer.state.best_model_checkpoint,
        'training_accuracy': accuracy_score(y_train, train_pred),
        'training_macro_f1': f1_score(y_train, train_pred, labels=LABELS, average='macro', zero_division=0),
        'validation_accuracy': accuracy_score(y_val, val_pred),
        'validation_macro_f1': f1_score(y_val, val_pred, labels=LABELS, average='macro', zero_division=0),
        'training_history': bert_trainer.state.log_history,
    }
    archive_path = Path('modeling_6_distilbert_results.zip')
    with zipfile.ZipFile(archive_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
        archive.writestr('metadata.json', json.dumps(export_metadata, indent=2))
        archive.writestr('training_predictions.csv', train_table.to_csv(index=False))
        archive.writestr('validation_predictions.csv', val_table.to_csv(index=False))
    print(f'Saved {archive_path} ({archive_path.stat().st_size / 2**20:.2f} MiB)')
    print('Validation macro F1:', round(export_metadata['validation_macro_f1'], 4))
    files.download(str(archive_path))
else:
    print('Local run: no BERT training; import the downloaded results in Section 5.')

## 5. Load Colab results into the local comparison

Put `modeling_6_distilbert_results.zip` in the same folder as this notebook. This cell reads the ZIP directly without extracting files. It checks the exact selected review IDs, labels, and SHA-256 fingerprints of review text against the local data. If they match, it adds DistilBERT training/validation scores and its validation confusion matrix to the same comparison used above. Rerunning the cell replaces the prior DistilBERT row rather than duplicating it. Until the ZIP is present, the three TF-IDF results remain visible.

In [6]:
import hashlib
import io
import json
import zipfile


def frame_fingerprint(frame):
    rows = [[int(row_id), str(row['color']), str(row['description'])]
            for row_id, row in frame.sort_index().iterrows()]
    payload = json.dumps(rows, ensure_ascii=False, separators=(',', ':')).encode('utf-8')
    return hashlib.sha256(payload).hexdigest()


def verified_predictions(table, frame, name):
    required = {'row_id', 'actual_color', 'predicted_color'}
    if set(table.columns) != required or table.empty or table.isna().any().any():
        raise ValueError(f'{name}: missing, extra, or empty prediction fields')
    if not table['row_id'].is_unique or set(table['row_id']) != set(frame.index):
        raise ValueError(f'{name}: row IDs do not match this notebook')
    aligned = table.set_index('row_id').reindex(frame.index)
    if not aligned['actual_color'].eq(frame['color']).all():
        raise ValueError(f'{name}: actual labels do not match this notebook')
    if not set(aligned['predicted_color']).issubset(LABELS):
        raise ValueError(f'{name}: unexpected predicted label')
    return aligned['predicted_color'].to_numpy()

archive_path = Path('modeling_6_distilbert_results.zip')
if archive_path.is_file():
    with zipfile.ZipFile(archive_path) as archive:
        required_files = {'metadata.json', 'training_predictions.csv', 'validation_predictions.csv'}
        if set(archive.namelist()) != required_files:
            raise ValueError('The ZIP does not contain the three expected Model 6 result files')
        exported = json.loads(archive.read('metadata.json'))
        saved_train_predictions = pd.read_csv(io.BytesIO(archive.read('training_predictions.csv')))
        saved_val_predictions = pd.read_csv(io.BytesIO(archive.read('validation_predictions.csv')))
    if exported.get('experiment') != 'modeling_6_distilbert' or exported.get('model') != 'distilbert/distilbert-base-uncased':
        raise ValueError('The ZIP is not from Model 6 DistilBERT')
    if exported.get('seed') != SEED or exported.get('training_rows') != len(train_sample) or exported.get('validation_rows') != len(validation_all):
        raise ValueError('The Colab run used different experiment settings or row counts')
    if exported.get('training_fingerprint') != frame_fingerprint(train_sample):
        raise ValueError('The Colab training reviews differ from this local sample')
    if exported.get('validation_fingerprint') != frame_fingerprint(validation_all):
        raise ValueError('The Colab validation reviews differ from this local validation set')
    bert_train_pred = verified_predictions(saved_train_predictions, train_sample, 'training')
    bert_val_pred = verified_predictions(saved_val_predictions, validation_all, 'validation')
    # Recompute from row-level predictions; do not trust saved summary numbers alone.
    score_rows = [row for row in score_rows if row['model'] != 'DistilBERT']
    validation_predictions.pop('DistilBERT', None)
    record_scores('DistilBERT', y_train, bert_train_pred, y_val, bert_val_pred)
    comparison = show_comparison()
    print('Colab checkpoint:', exported['best_checkpoint'])
    print('Epoch evaluations:')
    display(pd.DataFrame([entry for entry in exported['training_history'] if 'eval_macro_f1' in entry])
            [['epoch', 'eval_accuracy', 'eval_macro_f1']].round(4))
else:
    print(f'No {archive_path.name} yet. Download it from Colab, place it beside this notebook, and rerun this cell.')

,accuracy_training,accuracy_validation,macro_f1_training,macro_f1_validation,accuracy_gap,macro_f1_gap
model,,,,,,
DistilBERT,0.999,0.990,0.999,0.988,0.009,0.011
Linear SVM,1.000,0.989,1.000,0.987,0.010,0.013
Logistic regression,0.997,0.989,0.997,0.987,0.008,0.011
Naive Bayes,0.992,0.987,0.991,0.984,0.004,0.007



Naive Bayes: validation report
              precision    recall  f1-score   support

         red      0.996     0.986     0.991      6791
       white      0.966     0.989     0.977      2614

    accuracy                          0.987      9405
   macro avg      0.981     0.988     0.984      9405
weighted avg      0.987     0.987     0.987      9405



,predicted red,predicted white
actual red,6699,92
actual white,29,2585



Logistic regression: validation report
              precision    recall  f1-score   support

         red      0.996     0.989     0.992      6791
       white      0.971     0.990     0.981      2614

    accuracy                          0.989      9405
   macro avg      0.984     0.990     0.987      9405
weighted avg      0.989     0.989     0.989      9405



,predicted red,predicted white
actual red,6714,77
actual white,25,2589



Linear SVM: validation report
              precision    recall  f1-score   support

         red      0.996     0.989     0.993      6791
       white      0.973     0.989     0.981      2614

    accuracy                          0.989      9405
   macro avg      0.984     0.989     0.987      9405
weighted avg      0.989     0.989     0.989      9405



,predicted red,predicted white
actual red,6719,72
actual white,28,2586



DistilBERT: validation report
              precision    recall  f1-score   support

         red      0.996     0.991     0.993      6791
       white      0.977     0.989     0.983      2614

    accuracy                          0.990      9405
   macro avg      0.986     0.990     0.988      9405
weighted avg      0.990     0.990     0.990      9405



,predicted red,predicted white
actual red,6730,61
actual white,30,2584


Colab checkpoint: modeling_6_bert_checkpoints/checkpoint-1875
Epoch evaluations:


,epoch,eval_accuracy,eval_macro_f1
0,1.0,0.9838,0.9801
1,2.0,0.9897,0.9872
2,3.0,0.9903,0.9880


## Result and next decision

negligible overfit on training data. addition of more data to have negligible impact. 
Next steps: generalize to all red and white wines trained from just california validated on all, as well as trained on all validated on all. keep all four models for the moment, but seems like SVM and BERT are the main two. Bert slightly better predicting, but can draw more interpretation from SVMs